In [ ]:
import pandas as pd
import numpy as np
import json
import os

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq

In [ ]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.0 MB/s eta 0:00:00


In [ ]:
os.environ["GROQ_API_KEY"] = #enter your groq api key

In [ ]:
class SimpleEmbeddingMatcher:
    def __init__(self):
        self.model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

    def prepare_program_embeddings(self, program_texts):

        self.program_embeddings = self.model.encode(program_texts)
        self.program_texts = program_texts

    def find_best_match(self, user_query, top_k=5):
        # 1. Преобразуем запрос пользователя в эмбеддинг
        query_embedding = self.model.encode([user_query])

        # 2. Считаем косинусное сходство со всеми программами
        similarities = cosine_similarity(query_embedding, self.program_embeddings)[0]

        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                'program_text': self.program_texts[idx],
                'similarity_score': float(similarities[idx]),
                'program_index': int(idx)
            })

        return results

df = pd.read_csv('dataset_250.csv', sep=';')


In [ ]:
df.head()

,program_id,university,program_name,full_description,subjects,min_scores,program_type,specialization_focus,city,study_duration,degree_level
0,1,МГТУ им. Н.Э. Баумана,Программная инженерия,Системное программирование на C/C++ для аэроко...,"математика,русский,информатика","85,70,88",IT,Системное программирование,Москва,4,Бакалавриат
1,2,НИУ ВШЭ,Программная инженерия,Корпоративная разработка и enterprise-архитект...,"математика,русский,информатика","83,75,85",IT,Корпоративные приложения,Москва,4,Бакалавриат
2,3,ИТМО,Программная инженерия,Веб и мобильная разработка в продуктовых IT-ко...,"математика,русский,информатика","82,68,84",IT,Веб-разработка,Санкт-Петербург,4,Бакалавриат
3,4,МФТИ,Программная инженерия,Теоретическая информатика с углубленным изучен...,"математика,русский,информатика","84,72,86",IT,Алгоритмы и ML,Москва,4,Бакалавриат
4,5,СПбПУ Петра Великого,Программная инженерия,Промышленная автоматизация и разработка SCADA-...,"математика,русский,информатика","78,65,80",IT,Промышленный софт,Санкт-Петербург,4,Бакалавриат


## Взаимодействие с LLM для обработки запроса пользователя

In [ ]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def run_agent():
    messages = [
        {
            "role": "system",
            "content": (
                "Ты — агент по сбору карьерных предпочтений. Общайся с пользователем на русском языке. "
                "Твоя задача — получить три компонента: (1) список сдаваемых предметов (минимум 3!), (2) список баллов ЕГЭ, (3) описание карьерной цели. "
                "Пока у тебя нет всех данных — задавай уточняющие вопросы кратко и вежливо. "
                "Как только у тебя есть всё необходимое, ты ДОЛЖЕН вывести ТОЛЬКО один объект в формате JSON, "
                "без какого-либо дополнительного текста, пояснений, маркдауна, отступов или слов вроде 'Вот JSON, значит, у нас есть все необходимые данные'. "
                "Формат финального ответа: строго\n"
                "{\"subjects\": [\"предмет1\", ...], \"scores\": [оценка1, ...], \"text\": \"расширенное описание цели на русском\"}\n"
                "!Поле 'text' должно содержать слегка расширенное, но точное описание цели пользователя — без упоминания предметов и оценок. "
                "Никогда не выдумывай данные. Если данные противоречивы — проси уточнить. "
                "ВАЖНО: В финальном сообщении — ТОЛЬКО JSON. НИЧЕГО БОЛЬШЕ. Используй названия предметов именно в таком формате:{'биология', 'иностранный язык', 'информатика', 'история', 'литература', 'математика', 'обществознание', 'рисунок', 'русский', 'творческий экзамен', 'физика', 'химия'}"
            )
        }
    ]

    print("👋 Привет! Расскажи о своих предметах, оценках и карьере.")

    while True:
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            temperature=0.3,
            max_tokens=500
        )

        llm_reply = response.choices[0].message.content.strip()
        print(f"\n💬 Агент: {llm_reply}")

        try:
            parsed = json.loads(llm_reply)
            if (
                isinstance(parsed, dict) and
                set(parsed.keys()) == {"subjects", "scores", "text"} and
                isinstance(parsed["subjects"], list) and
                isinstance(parsed["scores"], list) and
                isinstance(parsed["text"], str) and
                len(parsed["subjects"]) == len(parsed["scores"]) and
                all(isinstance(s, str) for s in parsed["subjects"]) and
                all(isinstance(sc, (int, float)) for sc in parsed["scores"]) and
                len(parsed["text"]) > 10
            ):
                return parsed
        except (json.JSONDecodeError, TypeError, KeyError):
            pass

        user_input = input("\n👤 Вы: ").strip()
        if not user_input:
            user_input = "..."

        messages.append({"role": "assistant", "content": llm_reply})
        messages.append({"role": "user", "content": user_input})

req = run_agent()

👋 Привет! Расскажи о своих предметах, оценках и карьере.

💬 Агент: Какие предметы вы сдаёте (укажите минимум три из списка: биология, иностранный язык, информатика, история, литература, математика, обществознание, рисунок, русский, творческий экзамен, физика, химия)?

👤 Вы: русский

💬 Агент: Пожалуйста, укажите ещё минимум два предмета из списка (например, математика, физика, биология и т.д.).Какие ещё предметы вы сдаёте помимо русского? (нужно минимум три предмета в целом).Пожалуйста, назовите ещё два предмета, которые вы сдаёте, чтобы у нас было минимум три предмета.Какие ещё предметы из списка (биология, иностранный язык, информатика, история, литература, математика, обществознание, рисунок, творческий экзамен, физика, химия) вы сдаёте вместе с русским?Чтобы продолжить, мне нужны названия ещё минимум двух предметов, которые вы сдаёте. Пожалуйста, укажите их.Можете назвать ещё два предмета, которые вы сдаёте на ЕГЭ?Пожалуйста, уточните список предметов (минимум три). Это поможет собр

## Фильтрация направлений по проходным баллам

In [ ]:
def get_possible_df(df, input):
  subjects = input['subjects']
  scores = input['scores']
  d = dict()
  for i in range(len(subjects)):
    d[subjects[i]] = int(scores[i])
  mask = []
  for i in df.index:
    subj_t = list(map(str, df.loc[i, "subjects"].split(',')))
    scores_t = list(map(int, df.loc[i, "min_scores"].split(',')))
    flag = True
    for j in range(len(subj_t)):
      if subj_t[j] in d.keys():
        if scores_t[j] <= d[subj_t[j]]:
          continue
      flag = False
      break
    mask.append(flag)
  if sum(mask) == 0:
    df2 = pd.DataFrame({
        'program_id': ['-5', '-4', '-3', '-2', '-1'],
        'university': ['Синергия', 'Синергия', 'Синергия', 'Алубабу-политех', 'Алубабу-политех'],
        'program_name': ['Эстрадно-джазовое пение', 'Киберспорт', 'Event-менеджмент', 'Технология полиграфического и упаковочного производства', 'Хоббихорсинг'],
        'full_description': ['Направление, где тебя научат не просто петь, а «побеждать на музыкальном Олимпе»', 'Направление, где «играть в Dota 2» официально называется «тактико-стратегической подготовкой», а дисциплина «Экономика киберспортивных организаций» учит, как потратить призовые.', 'Description: Направление для тех, кто должен научиться организовывать ивенты уровня «закрытая вечеринка с Димашем»', 'Направление, где будущие инженеры печати и упаковки могут неофициально считаться специалистами по заворачиванию всего и вся в «алубабу»', 'Где ещё учить хоббихорсингу, как не в вузе, чьё название звучит как заклинание из детской считалочки?'],
        'subjects': ['', '', '', '', ''],
        'min_scores': ['0', '0', '0', '0', '0'],
        'program_type': ['TOP', 'TOP', 'TOP', 'TOP', 'TOP'],
        'specialization_focus': ['TOP', 'TOP', 'TOP', 'TOP', 'TOP'],
        'city': ['Москва', 'Москва', 'Москва', 'Елагуба', 'Елагуба'],
        'study_duration': ['4', '2', '3', '3', '4'],
        'degree_level': ['TOP', 'TOP', 'TOP', 'TOP', 'TOP'],

    })
    return 0, df2
  df2 = df[mask]
  return 1, df2

In [ ]:
possibles, df = get_possible_df(df, req)
possibles, len(df)

(1, 32)

In [ ]:
df['text_for_embedding'] = df['program_name'] + ". " + df['full_description']+"."+df['program_type'] +"." + df['specialization_focus']


program_texts = df['text_for_embedding'].tolist()


matcher = SimpleEmbeddingMatcher()
matcher.prepare_program_embeddings(program_texts)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
if possibles:
  results = matcher.find_best_match(req['text'], top_k=min(5, len(df)))
  i = 1
  for result in results:
      idx = result['program_index']
      program_info = df.iloc[idx]
      print(f"Приоритет: {i}")
      print(f"Вуз: {program_info['university']}")
      print(f"Программа: {program_info['program_name']}")
      print(f"Описание: {program_info['full_description']}")
      print(f"Город: {program_info['city']}")
      print("-" * 50)
      i += 1
else:
  for i in range(len(df)):
    print(f"Приоритет: {i+1}")
    print(f"Вуз: {df['university']}")
    print(f"Программа: {df['program_name']}")
    print(f"Описание: {df['full_description']}")
    print(f"Город: {df['city']}")
    print("-" * 50)


Приоритет: 1
Вуз: МГУ
Программа: Физика
Описание: Экспериментальная физика, физика твердого тела, нанотехнологии. Изучают методы измерений, работу на ускорителях, создание новых материалов. Выпускники работают в научных лабораториях, высокотехнологичных компаниях, исследовательских центрах
Город: Москва
--------------------------------------------------
Приоритет: 2
Вуз: МФТИ
Программа: Инноватика
Описание: Научно-технологические инновации, коммерциализация научных разработок, deep tech стартапы. Изучают коммерциализацию научных разработок, создание deep tech стартапов, управление научно-технологическими проектами, интеллектуальную собственность. Карьера в научно-технологических стартапах, технопарках, венчурных фондах, специализирующихся на deep tech
Город: Москва
--------------------------------------------------
Приоритет: 3
Вуз: МГУ
Программа: Геофизика
Описание: Сейсмология, гравиметрия, магнитометрия, разведочная геофизика, геофизические методы исследований. Изучают сейсмологию, 